# Full Cached Voice Prediction Pipeline

This notebook runs the **final real-time gesture → voice pipeline** using:

1. Saved gesture model (`CombinedGesture_model.pkl`, `Modelscaler.pkl`, `label_encoder.pkl`)
2. Training dataset (`numbers_letters_dataset_sorted.csv`)
3. **Approved cached MP3 files** under `generated_audio/.../chosen/`

Flow: **gesture input → scale → predict → decode label → resolve approved MP3 → copy/play → save under `predictions/`**

Audio was previously generated with Edge TTS, listened to, and curated manually. **This notebook does not call Edge TTS, recut audio, or create new pronunciations** — it only uses the approved files.

All paths are **relative** (`pathlib`) so the project folder can be shared or moved.

## Imports and Relative Paths

Import libraries and define project-relative folders (`BASE_DIR`, models, dataset, generated audio, predictions output).

In [1]:
import re
import shutil
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Audio, display

BASE_DIR = Path(".")
MODEL_PATH = BASE_DIR / "CombinedGesture_model.pkl"
SCALER_PATH = BASE_DIR / "Modelscaler.pkl"
ENCODER_PATH = BASE_DIR / "label_encoder.pkl"
DATASET_PATH = BASE_DIR / "numbers_letters_dataset_sorted.csv"

GENERATED_AUDIO_DIR = BASE_DIR / "generated_audio"
LETTERS_CHOSEN_DIR = GENERATED_AUDIO_DIR / "edge_tts_mapping_v2" / "chosen"
NUMBERS_CHOSEN_DIR = GENERATED_AUDIO_DIR / "edge_tts_numbers_mapping_v1" / "chosen"
PREDICTIONS_DIR = BASE_DIR / "predictions"

FEATURE_COLS = [
    "flex1", "flex2", "flex3", "flex4", "flex5",
    "accX", "accY", "accZ", "gyroX", "gyroY", "gyroZ",
]

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR.resolve())
print("letters chosen:", LETTERS_CHOSEN_DIR)
print("numbers chosen:", NUMBERS_CHOSEN_DIR)
print("predictions out:", PREDICTIONS_DIR)

BASE_DIR: C:\Users\sondo\Desktop\Voice Model Stuff
letters chosen: generated_audio\edge_tts_mapping_v2\chosen
numbers chosen: generated_audio\edge_tts_numbers_mapping_v1\chosen
predictions out: predictions


## Load Saved Gesture Model

Load the trained classifier, feature scaler, and label encoder from disk.

In [2]:
clf = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
label_encoder = joblib.load(ENCODER_PATH)

print("✅ model loaded")
print("✅ scaler loaded")
print("✅ encoder loaded")
print(f"classes: {len(label_encoder.classes_)}")

✅ model loaded
✅ scaler loaded
✅ encoder loaded
classes: 39


## Load Dataset

Load `numbers_letters_dataset_sorted.csv` for sample predictions and inspection.

In [3]:
dataset = pd.read_csv(DATASET_PATH)
print("shape:", dataset.shape)
print("columns:", list(dataset.columns))
display(dataset.head(5))

shape: (7800, 12)
columns: ['flex1', 'flex2', 'flex3', 'flex4', 'flex5', 'accX', 'accY', 'accZ', 'gyroX', 'gyroY', 'gyroZ', 'label']


,flex1,flex2,flex3,flex4,flex5,accX,accY,accZ,gyroX,gyroY,gyroZ,label
0,861,476,630,592,671,-8392,14164,404,-606,-1314,272,أ
1,862,468,630,591,677,-8392,14164,404,-606,-1314,272,أ
2,864,470,630,591,679,-8392,14164,404,-606,-1314,272,أ
3,873,467,631,592,679,-8392,14164,404,-606,-1314,272,أ
4,865,469,631,592,678,-8392,14164,404,-606,-1314,272,أ


## Build Cached Audio Lookup Tables

Map each letter (ا–ي) and number digit (0–10) to an **approved** MP3 under the `chosen/` folders. Paths are stored relative to `BASE_DIR`.

In [4]:
APPROVED_LETTER_FILES = [
    {"label": "ا", "approved_filename": "letter_ا_alif.mp3"},
    {"label": "ب", "approved_filename": "letter_ب_cut_0.45.mp3"},
    {"label": "ت", "approved_filename": "letter_ت_teh.mp3"},
    {"label": "ث", "approved_filename": "comp_seh_ث_ث_ه.mp3"},
    {"label": "ج", "approved_filename": "letter_ج_geem.mp3"},
    {"label": "ح", "approved_filename": "letter_ح_ha.mp3"},
    {"label": "خ", "approved_filename": "letter_خ_kha.mp3"},
    {"label": "د", "approved_filename": "comp_dal_د_دال_bang.mp3"},
    {"label": "ذ", "approved_filename": "comp_zal_ذ_ذال_bang.mp3"},
    {"label": "ر", "approved_filename": "letter_ر_ra.mp3"},
    {"label": "ز", "approved_filename": "comp_zay_ز_zeen_latin_long.mp3"},
    {"label": "س", "approved_filename": "comp_seen_س_seen_split_cut.mp3"},
    {"label": "ش", "approved_filename": "letter_ش_sheen.mp3"},
    {"label": "ص", "approved_filename": "comp_sad_ص_saad_long_sukoon.mp3"},
    {"label": "ض", "approved_filename": "comp_dad_ض_daad_long_sukoon.mp3"},
    {"label": "ط", "approved_filename": "comp_ta_ط_ط_ه.mp3"},
    {"label": "ظ", "approved_filename": "letter_ظ_dha.mp3"},
    {"label": "ع", "approved_filename": "letter_ع_ein.mp3"},
    {"label": "غ", "approved_filename": "letter_غ_ghein.mp3"},
    {"label": "ف", "approved_filename": "comp_faa_ف_ف_ه.mp3"},
    {"label": "ق", "approved_filename": "comp_qaf_ق_qaf_split_cut_v2.mp3"},
    {"label": "ك", "approved_filename": "comp_kaf_ك_كاف_sukoon.mp3"},
    {"label": "ل", "approved_filename": "comp_lam_ل_lam_latin.mp3"},
    {"label": "م", "approved_filename": "letter_م_meem.mp3"},
    {"label": "ن", "approved_filename": "letter_ن_noon.mp3"},
    {"label": "ه", "approved_filename": "comp_ha_ه_ه_ه.mp3"},
    {"label": "و", "approved_filename": "comp_waw_و_واو_diac.mp3"},
    {"label": "ي", "approved_filename": "letter_ي_yeh.mp3"},
]

APPROVED_NUMBER_FILES = [
    {"label": "0", "approved_filename": "num_00_v02.mp3"},
    {"label": "1", "approved_filename": "num_01.mp3"},
    {"label": "2", "approved_filename": "num_02_atneen_bang_cut_0_59.mp3"},
    {"label": "3", "approved_filename": "num_03.mp3"},
    {"label": "4", "approved_filename": "num_04.mp3"},
    {"label": "5", "approved_filename": "num_05.mp3"},
    {"label": "6", "approved_filename": "num_06.mp3"},
    {"label": "7", "approved_filename": "num_07.mp3"},
    {"label": "8", "approved_filename": "num_08.mp3"},
    {"label": "9", "approved_filename": "num_09.mp3"},
    {"label": "10", "approved_filename": "num_10_asharah_sukoon.mp3"},
]

def _rel_chosen_path(chosen_dir: Path, filename: str) -> str:
    return (chosen_dir / filename).relative_to(BASE_DIR).as_posix()

LETTER_AUDIO_MAP = {
    row["label"]: _rel_chosen_path(LETTERS_CHOSEN_DIR, row["approved_filename"])
    for row in APPROVED_LETTER_FILES
}

NUMBER_AUDIO_MAP = {
    row["label"]: _rel_chosen_path(NUMBERS_CHOSEN_DIR, row["approved_filename"])
    for row in APPROVED_NUMBER_FILES
}

# Model may predict أ; approved letter audio is keyed as ا
LETTER_LABEL_ALIASES = {"أ": "ا", "آ": "ا", "إ": "ا", "ٱ": "ا"}

# Model predicts Arabic number words; audio map uses digit keys 0–10
NUMBER_WORD_TO_DIGIT = {
    "صفر": "0",
    "واحد": "1",
    "اثنين": "2",
    "ثلاثة": "3",
    "أربعة": "4",
    "خمسة": "5",
    "ستة": "6",
    "سبعة": "7",
    "ثمانية": "8",
    "تسعة": "9",
    "عشرة": "10",
}

missing_letters = [k for k, v in LETTER_AUDIO_MAP.items() if not (BASE_DIR / v).is_file()]
missing_numbers = [k for k, v in NUMBER_AUDIO_MAP.items() if not (BASE_DIR / v).is_file()]
if missing_letters:
    raise FileNotFoundError(f"Missing letter audio: {missing_letters}")
if missing_numbers:
    raise FileNotFoundError(f"Missing number audio: {missing_numbers}")

print(f"letter audio entries: {len(LETTER_AUDIO_MAP)}")
print(f"number audio entries: {len(NUMBER_AUDIO_MAP)}")
print("sample letter:", list(LETTER_AUDIO_MAP.items())[:3])
print("sample number:", list(NUMBER_AUDIO_MAP.items())[:3])

letter audio entries: 28
number audio entries: 11
sample letter: [('ا', 'generated_audio/edge_tts_mapping_v2/chosen/letter_ا_alif.mp3'), ('ب', 'generated_audio/edge_tts_mapping_v2/chosen/letter_ب_cut_0.45.mp3'), ('ت', 'generated_audio/edge_tts_mapping_v2/chosen/letter_ت_teh.mp3')]
sample number: [('0', 'generated_audio/edge_tts_numbers_mapping_v1/chosen/num_00_v02.mp3'), ('1', 'generated_audio/edge_tts_numbers_mapping_v1/chosen/num_01.mp3'), ('2', 'generated_audio/edge_tts_numbers_mapping_v1/chosen/num_02_atneen_bang_cut_0_59.mp3')]


## Prediction Function

Scale glove sensor values, run the classifier, and decode the predicted class label. Uses a pandas `DataFrame` with scaler feature names when available.

In [5]:
def predict_gesture(sensor_values):
    """Predict gesture label from 11 glove sensor readings."""
    values = list(sensor_values)
    if len(values) != len(FEATURE_COLS):
        raise ValueError(f"Expected {len(FEATURE_COLS)} values, got {len(values)}")

    if hasattr(scaler, "feature_names_in_"):
        X = pd.DataFrame([values], columns=list(scaler.feature_names_in_))
    else:
        X = np.array(values, dtype=np.float64).reshape(1, -1)

    X_scaled = scaler.transform(X)
    pred_code = clf.predict(X_scaled)[0]
    return str(label_encoder.inverse_transform([pred_code])[0])


test_row = dataset.iloc[0][FEATURE_COLS].tolist()
print("smoke test prediction:", predict_gesture(test_row), "| expected:", dataset.iloc[0]["label"])

smoke test prediction: أ | expected: أ


## Cached Audio Resolver

Given a predicted label, return the relative path to the approved cached MP3 (letter or number), or `None` if not found.

In [6]:
def get_cached_audio_for_prediction(label: str):
    """Resolve approved cached MP3 path for a model label."""
    tok = str(label).strip()
    if not tok:
        return None

    letter_key = LETTER_LABEL_ALIASES.get(tok, tok)
    if letter_key in LETTER_AUDIO_MAP:
        return LETTER_AUDIO_MAP[letter_key]

    if tok in NUMBER_AUDIO_MAP:
        return NUMBER_AUDIO_MAP[tok]

    digit_key = NUMBER_WORD_TO_DIGIT.get(tok)
    if digit_key and digit_key in NUMBER_AUDIO_MAP:
        return NUMBER_AUDIO_MAP[digit_key]

    warnings.warn(f"No approved cached audio for label: {tok!r}")
    return None


for sample in ["أ", "ب", "اثنين", "خمسة", "10"]:
    print(sample, "->", get_cached_audio_for_prediction(sample))

أ -> generated_audio/edge_tts_mapping_v2/chosen/letter_ا_alif.mp3
ب -> generated_audio/edge_tts_mapping_v2/chosen/letter_ب_cut_0.45.mp3
اثنين -> generated_audio/edge_tts_numbers_mapping_v1/chosen/num_02_atneen_bang_cut_0_59.mp3
خمسة -> generated_audio/edge_tts_numbers_mapping_v1/chosen/num_05.mp3
10 -> generated_audio/edge_tts_numbers_mapping_v1/chosen/num_10_asharah_sukoon.mp3


## Save Prediction Audio

Copy the approved source MP3 into `predictions/` with a stable, sanitized filename.

In [7]:
_prediction_save_index = 0
SAVED_PREDICTION_PATHS = []


def _sanitize_label_for_filename(label: str) -> str:
    safe = re.sub(r'[<>:"/\\|?*]', "_", str(label).strip())
    return safe or "unknown"


def save_prediction_audio(source_audio_path, prediction_label, index=None):
    global _prediction_save_index
    if index is None:
        _prediction_save_index += 1
        index = _prediction_save_index

    src = Path(source_audio_path)
    if not src.is_file():
        src = BASE_DIR / source_audio_path
    if not src.is_file():
        raise FileNotFoundError(f"Source audio not found: {source_audio_path}")

    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
    label_safe = _sanitize_label_for_filename(prediction_label)
    dest = PREDICTIONS_DIR / f"prediction_{index:02d}_{label_safe}.mp3"
    shutil.copy2(src, dest)
    rel = dest.relative_to(BASE_DIR).as_posix()
    SAVED_PREDICTION_PATHS.append(rel)
    return rel


def play_cached_audio(relative_path):
    path = BASE_DIR / relative_path
    if path.is_file():
        display(Audio(filename=str(path)))
    else:
        print("Missing audio:", relative_path)

## Dataset Prediction Examples

Run predictions on several dataset rows: show input, expected label, prediction, resolved MP3, copy to `predictions/`, and play audio.

In [8]:
SAMPLE_LABELS = ["أ", "ل", "ت", "أربعة", "ستة", "خمسة"]
dataset_examples = []
for lbl in SAMPLE_LABELS:
    matches = dataset[dataset["label"] == lbl]
    if len(matches):
        dataset_examples.append(matches.iloc[0])

print(f"Running {len(dataset_examples)} dataset examples\n")

for i, row in enumerate(dataset_examples, start=1):
    sensor = row[FEATURE_COLS].tolist()
    expected = row["label"]
    predicted = predict_gesture(sensor)
    audio_rel = get_cached_audio_for_prediction(predicted)

    print("=" * 50)
    print(f"Example {i}")
    print("input:", sensor)
    print("expected:", expected)
    print("predicted:", predicted)
    print("cached audio:", audio_rel)

    if audio_rel:
        saved = save_prediction_audio(audio_rel, predicted, index=i)
        print("saved:", saved)
        play_cached_audio(saved)
    print()

Running 6 dataset examples

Example 1
input: [np.int64(861), np.int64(476), np.int64(630), np.int64(592), np.int64(671), np.int64(-8392), np.int64(14164), np.int64(404), np.int64(-606), np.int64(-1314), np.int64(272)]
expected: أ
predicted: أ
cached audio: generated_audio/edge_tts_mapping_v2/chosen/letter_ا_alif.mp3
saved: predictions/prediction_01_أ.mp3


Example 2
input: [np.int64(758), np.int64(841), np.int64(320), np.int64(540), np.int64(465), np.int64(-11292), np.int64(3704), np.int64(-11276), np.int64(433), np.int64(-1603), np.int64(-66)]
expected: ل
predicted: ل
cached audio: generated_audio/edge_tts_mapping_v2/chosen/comp_lam_ل_lam_latin.mp3
saved: predictions/prediction_02_ل.mp3


Example 3
input: [np.int64(595), np.int64(848), np.int64(704), np.int64(775), np.int64(682), np.int64(-7632), np.int64(7312), np.int64(-12772), np.int64(-278), np.int64(-1928), np.int64(-180)]
expected: ت
predicted: ت
cached audio: generated_audio/edge_tts_mapping_v2/chosen/letter_ت_teh.mp3
saved: predictions/prediction_03_ت.mp3


Example 4
input: [np.int64(925), np.int64(1305), np.int64(945), np.int64(806), np.int64(966), np.int64(2372), np.int64(16252), np.int64(-1332), np.int64(-350), np.int64(-2148), np.int64(-672)]
expected: أربعة
predicted: أربعة
cached audio: generated_audio/edge_tts_numbers_mapping_v1/chosen/num_04.mp3
saved: predictions/prediction_04_أربعة.mp3


Example 5
input: [np.int64(883), np.int64(603), np.int64(613), np.int64(482), np.int64(558), np.int64(2300), np.int64(16396), np.int64(892), np.int64(73), np.int64(267), np.int64(-371)]
expected: ستة
predicted: ستة
cached audio: generated_audio/edge_tts_numbers_mapping_v1/chosen/num_06.mp3
saved: predictions/prediction_05_ستة.mp3


Example 6
input: [np.int64(895), np.int64(902), np.int64(944), np.int64(809), np.int64(958), np.int64(1876), np.int64(16480), np.int64(504), np.int64(140), np.int64(415), np.int64(-444)]
expected: خمسة
predicted: خمسة
cached audio: generated_audio/edge_tts_numbers_mapping_v1/chosen/num_05.mp3
saved: predictions/prediction_06_خمسة.mp3


## Manual Gloves Input Tests

Real glove sensor arrays (11 values: 5 flex + 3 accel + 3 gyro). Sources:

- `CombinedDatasetModel.ipynb` demo cells
- Representative rows from `numbers_letters_dataset_sorted.csv`

Each test: predict → resolve **approved cached MP3** (no Edge TTS) → copy to `predictions/` → playback.



In [9]:
def run_manual_glove_test(test_num: int, gloves_input: list, expected_label=None):
    """Run one manual glove test: predict, cache lookup, save, playback."""
    global _prediction_save_index

    predicted = predict_gesture(gloves_input)
    audio_rel = get_cached_audio_for_prediction(predicted)

    print(f"Test #{test_num}")
    print("Input numbers:", gloves_input)
    if expected_label is not None:
        print("Expected label:", expected_label)
    print("Predicted label:", predicted)
    print("Cached audio used:", audio_rel if audio_rel else "(none)")

    saved_rel = None
    if audio_rel:
        _prediction_save_index += 1
        saved_rel = save_prediction_audio(audio_rel, predicted, index=_prediction_save_index)
        print("Saved prediction audio:", saved_rel)
        play_cached_audio(saved_rel)
    print()

    return {
        "test_num": test_num,
        "input": gloves_input,
        "expected": expected_label,
        "predicted": predicted,
        "cached_audio": audio_rel,
        "saved": saved_rel,
    }


MANUAL_GLOVE_TEST_RESULTS = []

# Continue file numbering after dataset examples (indices 1..len(dataset_examples))
_prediction_save_index = max(_prediction_save_index, len(dataset_examples))

# --- Test #1: أ (CombinedDatasetModel.ipynb) ---
GlovesInput = [
    865, 472, 632, 593, 675,
    -8380, 14150, 410,
    -610, -1310, 275,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(1, GlovesInput, expected_label="أ")
)

# --- Test #2: أربعة (numbers_letters_dataset_sorted) ---
GlovesInput = [
    925, 1305, 945, 806, 966,
    2372, 16252, -1332,
    -350, -2148, -672,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(2, GlovesInput, expected_label="أربعة")
)

# --- Test #3: خمسة (CombinedDatasetModel.ipynb) ---
GlovesInput = [
    910, 885, 945, 805, 960,
    1700, 16450, 450,
    100, -150, -400,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(3, GlovesInput, expected_label="خمسة")
)

# --- Test #4: خ (numbers_letters_dataset_sorted) ---
GlovesInput = [
    869, 805, 752, 910, 882,
    -12820, 5648, -7832,
    -1118, -2214, -670,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(4, GlovesInput, expected_label="خ")
)

# --- Test #5: ت (numbers_letters_dataset_sorted) ---
GlovesInput = [
    595, 848, 704, 775, 682,
    -7632, 7312, -12772,
    -278, -1928, -180,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(5, GlovesInput, expected_label="ت")
)

# --- Test #6: ل (numbers_letters_dataset_sorted) ---
GlovesInput = [
    758, 841, 320, 540, 465,
    -11292, 3704, -11276,
    433, -1603, -66,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(6, GlovesInput, expected_label="ل")
)

# --- Test #7: واحد (numbers_letters_dataset_sorted) ---
GlovesInput = [
    945, 699, 625, 464, 935,
    -464, 16972, 756,
    695, -105, 270,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(7, GlovesInput, expected_label="واحد")
)

# --- Test #8: ثلاثة (numbers_letters_dataset_sorted) ---
GlovesInput = [
    922, 325, 940, 791, 963,
    424, 16508, -1416,
    295, 1871, -794,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(8, GlovesInput, expected_label="ثلاثة")
)

# --- Test #9: سبعة (numbers_letters_dataset_sorted) ---
GlovesInput = [
    893, 671, 611, 383, 924,
    1816, 16344, 16,
    140, 258, -708,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(9, GlovesInput, expected_label="سبعة")
)

# --- Test #10: عشرة (numbers_letters_dataset_sorted) ---
GlovesInput = [
    923, 563, 560, 383, 895,
    -6684, 14208, -4292,
    79, 284, -407,
]
MANUAL_GLOVE_TEST_RESULTS.append(
    run_manual_glove_test(10, GlovesInput, expected_label="عشرة")
)

print(f"Manual glove tests completed: {len(MANUAL_GLOVE_TEST_RESULTS)}")



Test #1
Input numbers: [865, 472, 632, 593, 675, -8380, 14150, 410, -610, -1310, 275]
Expected label: أ
Predicted label: أ
Cached audio used: generated_audio/edge_tts_mapping_v2/chosen/letter_ا_alif.mp3
Saved prediction audio: predictions/prediction_07_أ.mp3


Test #2
Input numbers: [925, 1305, 945, 806, 966, 2372, 16252, -1332, -350, -2148, -672]
Expected label: أربعة
Predicted label: أربعة
Cached audio used: generated_audio/edge_tts_numbers_mapping_v1/chosen/num_04.mp3
Saved prediction audio: predictions/prediction_08_أربعة.mp3


Test #3
Input numbers: [910, 885, 945, 805, 960, 1700, 16450, 450, 100, -150, -400]
Expected label: خمسة
Predicted label: خمسة
Cached audio used: generated_audio/edge_tts_numbers_mapping_v1/chosen/num_05.mp3
Saved prediction audio: predictions/prediction_09_خمسة.mp3


Test #4
Input numbers: [869, 805, 752, 910, 882, -12820, 5648, -7832, -1118, -2214, -670]
Expected label: خ
Predicted label: خ
Cached audio used: generated_audio/edge_tts_mapping_v2/chosen/letter_خ_kha.mp3
Saved prediction audio: predictions/prediction_10_خ.mp3


Test #5
Input numbers: [595, 848, 704, 775, 682, -7632, 7312, -12772, -278, -1928, -180]
Expected label: ت
Predicted label: ت
Cached audio used: generated_audio/edge_tts_mapping_v2/chosen/letter_ت_teh.mp3
Saved prediction audio: predictions/prediction_11_ت.mp3


...

Test #7
Input numbers: [945, 699, 625, 464, 935, -464, 16972, 756, 695, -105, 270]
Expected label: واحد
Predicted label: واحد
Cached audio used: generated_audio/edge_tts_numbers_mapping_v1/chosen/num_01.mp3
Saved prediction audio: predictions/prediction_13_واحد.mp3


Test #8
Input numbers: [922, 325, 940, 791, 963, 424, 16508, -1416, 295, 1871, -794]
Expected label: ثلاثة
Predicted label: ثلاثة
Cached audio used: generated_audio/edge_tts_numbers_mapping_v1/chosen/num_03.mp3
Saved prediction audio: predictions/prediction_14_ثلاثة.mp3


Test #9
Input numbers: [893, 671, 611, 383, 924, 1816, 16344, 16, 140, 258, -708]
Expected label: سبعة
Predicted label: سبعة
Cached audio used: generated_audio/edge_tts_numbers_mapping_v1/chosen/num_07.mp3
Saved prediction audio: predictions/prediction_15_سبعة.mp3


Test #10
Input numbers: [923, 563, 560, 383, 895, -6684, 14208, -4292, 79, 284, -407]
Expected label: عشرة
Predicted label: عشرة
Cached audio used: generated_audio/edge_tts_numbers_mapping_v1/chosen/num_10_asharah_sukoon.mp3
Saved prediction audio: predictions/prediction_16_عشرة.mp3



Manual glove tests completed: 10


## Final Verification

Confirm relative paths, loaded assets, cached audio maps, and prediction outputs.

In [10]:
letter_files_on_disk = list(LETTERS_CHOSEN_DIR.glob("*.mp3"))
number_files_on_disk = list(NUMBERS_CHOSEN_DIR.glob("*.mp3"))
prediction_files = list(PREDICTIONS_DIR.glob("prediction_*.mp3"))

checks = [
    ("relative paths working", not MODEL_PATH.is_absolute() and not LETTERS_CHOSEN_DIR.is_absolute()),
    ("model loaded", clf is not None),
    ("scaler loaded", scaler is not None),
    ("encoder loaded", label_encoder is not None),
    ("dataset loaded", len(dataset) > 0),
    ("cached audio loaded", len(LETTER_AUDIO_MAP) > 0 and len(NUMBER_AUDIO_MAP) > 0),
    ("predictions generated", len(prediction_files) > 0),
    ("prediction audio copied", all(p.is_file() for p in prediction_files)),
]

for name, ok in checks:
    print(("✅" if ok else "❌"), name)

print("✅ playback widgets visible (run cells above to hear audio)")
print()
print("letter audio files in chosen/:", len(letter_files_on_disk))
print("number audio files in chosen/:", len(number_files_on_disk))
print("prediction outputs saved:", len(prediction_files))
print("predictions folder:", PREDICTIONS_DIR.relative_to(BASE_DIR).as_posix())
for p in sorted(prediction_files)[:15]:
    print(" -", p.name)
if len(prediction_files) > 15:
    print(f" ... and {len(prediction_files) - 15} more")

✅ relative paths working
✅ model loaded
✅ scaler loaded
✅ encoder loaded
✅ dataset loaded
✅ cached audio loaded
✅ predictions generated
✅ prediction audio copied
✅ playback widgets visible (run cells above to hear audio)

letter audio files in chosen/: 28
number audio files in chosen/: 11
prediction outputs saved: 36
predictions folder: predictions
 - prediction_01_أ.mp3
 - prediction_02_اثنين.mp3
 - prediction_02_ب.mp3
 - prediction_02_ل.mp3
 - prediction_03_ت.mp3
 - prediction_03_خمسة.mp3
 - prediction_04_أربعة.mp3
 - prediction_04_اثنين.mp3
 - prediction_04_ب.mp3
 - prediction_05_ت.mp3
 - prediction_05_خمسة.mp3
 - prediction_05_ستة.mp3
 - prediction_06_تسعة.mp3
 - prediction_06_ج.mp3
 - prediction_06_خمسة.mp3
 ... and 21 more
